In [1]:
import numpy as np


from qiskit import QuantumCircuit,  QuantumRegister, ClassicalRegister, AncillaRegister, transpile, qasm2
from qiskit.circuit import Parameter
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit.circuit.library import PauliEvolutionGate, hamiltonian_variational_ansatz
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit.synthesis import SuzukiTrotter
from qiskit.primitives import BackendEstimatorV2 as Estimator

import os

import quimb.tensor as qtn
import quimb as qu

import csv
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib.ticker as tck

In this Jupyter notebook, I use the quimb tensor network package to approximate long TFIM chains, and then use the properties of MPS to evolve and measure the state in order to calculate the dichotomic time correlator $C_Q(t) = \langle \psi |\frac{1}{2} \{e^{i H t} Q e^{ - i H t}, Q \} | \psi \rangle $

In [171]:
def apply_gate(psi, measure_opp = 'Z', measure_site = 0):
    """Applies the measure_opp gate to the measure_site tensor of psi and returns a new psi"""
    
    L = len(psi.tensors)
    psi_Q = psi.copy()
    
    if len(measure_opp) == 1:
        Opp =qu.pauli(measure_opp)
        # Apply X gate to site n
        psi_Q = psi_Q.gate(Opp, measure_site, contract='swap+split')

    elif measure_opp == "X_even":

        for i in range(0, L, 2):
            Opp = qu.pauli('X')
            psi_Q = psi_Q.gate(Opp, i, contract='swap+split')

    return psi_Q


def DMRG_ising_GS(L, J = 1, h = 1):
    """Approxmate the ground state of the Ising model.
    Returns state, energy, and quimb.tensor.tn1d.tebd.LocalHam1D 
    class object suitable for TEBD evolution"""
    H_ising = qtn.SpinHam1D(S=1/2, cyclic=False)
    H_ising += -1.0 * J , 'Z', 'Z'
    H_ising += -1.0 * h , 'X'
    H_MPO = H_ising.build_mpo( L )
    H_ising_LocalHam = H_ising.build_local_ham( L )

    #run the DMRG procedure
    dmrg = qtn.DMRG2(H_MPO)
    dmrg.solve(max_sweeps=20, verbosity=0, cutoffs = 1e-09 )

    #call the ground state as a tensor network
    return dmrg.state, dmrg.energy, H_ising_LocalHam


def TFIM_DMRG_correlation(L, J = 1, h = 1, time_end = 2, time_step = 0.05, measure_site = 0, measure_opp = 'X', dt = 0.05, time_start = 0):
    """Calcuates C_Q(t) for ground state of Ising Hamiltonian 
    for t in time_range and outputs an array of C_Q(t). """

    if measure_site not in range(L):
        raise ValueError("measure_site not in the chain")
    elif  time_step * 10000 %1 != 0:
        raise ValueError("time_step * 1e4 should be a whole number for ease of calculating K")

    #call the ground state as a tensor network
    psi0, E0, H_ising_LocalHam = DMRG_ising_GS(L, J = J, h = h)

    #apply the measurement gate
    psi_Q = apply_gate(psi0, measure_opp = measure_opp, measure_site = measure_site)

    #The evolution is given by \exp(- i H t) | \psi >
    #time evolve for every time in time_range
    tebd = qtn.TEBD( psi_Q, H_ising_LocalHam, dt=dt  )

    correlation_C = []
    time_range = time_range = [round(t, 4) for t in np.arange(time_start, time_end, time_step)]
    for t in time_range:
        tebd.update_to(T = t, progbar = False)
        psi_t = tebd.pt
        psi_t = apply_gate(psi_t, measure_opp = measure_opp, measure_site = measure_site)
        Cor_C = psi0.H @ psi_t
        Cor_C *= np.exp(1j * E0 * t)
        Cor_C = np.real(Cor_C)
        correlation_C += [Cor_C]

    #eliminate machine-level precision error

    return time_range,  correlation_C

def TFIM_DMRG_LG_K(L, J = 1, h = 1, time_end = 2, time_step = 0.05, measure_site = 0, measure_opp = 'X', dt = 0.05):
    """Calculates the Legget-Garg correlation K = 2C(t) - C(2t)"""
    
    #Calculates C values from 0 to time_end
    times_1, leg_1 = TFIM_DMRG_correlation(L, J = J, h = h, time_end = time_end, time_step = time_step, measure_site = measure_site, measure_opp = measure_opp , dt = dt)
    #Calculates only the extra C values needed to calculate K
    times_2, leg_2 = TFIM_DMRG_correlation(L, J = J, h = h, time_end = 2 * time_end, time_start = time_end  ,time_step = 2* time_step, measure_site = measure_site, measure_opp = measure_opp , dt = dt)
    time_tot = times_1 + times_2
    cor_tot = leg_1 + leg_2

    K_lst = []
    # K = 2C(t) - C(2t)
    for i, t in enumerate(times_1):
        K_lst +=[leg_1[i] * 2 - cor_tot[time_tot.index(t* 2)]]
        
    return K_lst


psi = TFIM_DMRG_LG_K(8, J = 1, h= .1, measure_site = 7, measure_opp = "X", time_step = .1, time_end = 3)
psi


[np.float64(0.9999999999999966),
 np.float64(1.00249605229466),
 np.float64(1.0099369187640603),
 np.float64(1.022181320209508),
 np.float64(1.0389957571826385),
 np.float64(1.060057433212135),
 np.float64(1.084958296927637),
 np.float64(1.113210152680314),
 np.float64(1.1442507771881392),
 np.float64(1.1774509669580793),
 np.float64(1.2121224302659157),
 np.float64(1.247526427309347),
 np.float64(1.2828830543747411),
 np.float64(1.3173810579190688),
 np.float64(1.350188062965807),
 np.float64(1.3804601198299786),
 np.float64(1.4073559936417097),
 np.float64(1.4300428536172758),
 np.float64(1.4477100411349066),
 np.float64(1.4595787591633236),
 np.float64(1.464912162658515),
 np.float64(1.4630249997611409),
 np.float64(1.4532926925576977),
 np.float64(1.4351597555410758),
 np.float64(1.4081474543484096),
 np.float64(1.371860621495589),
 np.float64(1.3259935503477849),
 np.float64(1.2703349039669667),
 np.float64(1.2047715844487876),
 np.float64(1.1292915191699555)]